In [1]:
# Ingerir PDF
import pdfplumber
import pandas as pd
import re
import numpy as np
import re

Por que não está gerando a tabela? Descubra sozinha antes

## 13. Extração de dados para Cartão de Crédito

In [2]:
# Refere-se a seção 13

with pdfplumber.open("Demonstrações Financeiras 3T25.pdf") as pdf:
    page = pdf.pages[17]  # página 18 (índice começa em 0)
    text = page.extract_text()

# Remove espaços em branco extras de cada linha ( pré-processamento )
text = "\n".join(line.strip() for line in text.splitlines())

# Define início ( start ) e fim ( end ) do trecho a ser extraído. 
start = text.find("c) Provisão para perdas de crédito - por qualidade de crédito vs. estágios")
end = text.find("Total", start)

trecho = text[start:end]
print(trecho)

# divide em linhas para facilitar a criação do dataframe
linhas = [l.strip() for l in trecho.splitlines() if l.strip()]

c) Provisão para perdas de crédito - por qualidade de crédito vs. estágios
30/09/2025 31/12/2024
Provisão Índice de Provisão Índice de
Exposição Exposição
% para perdas % cobertura % para perdas % cobertura
bruta bruta
de crédito (%) de crédito (%)
Forte (PD < 5%) 9.202.509 45,9% 193.814 5,9% 2,1% 6.644.920 45,5% 126.401 5,3% 1,9%
Estágio 1 9.202.497 100,0% 193.814 100,0% 2,1% 6.628.863 99,8% 126.147 99,8% 1,9%
Estágio 2 12 – – – – 16.057 0,2% 254 0,2% 1,6%
Satisfatório
6.063.886 30,3% 509.759 15,4% 8,7% 4.304.062 29,4% 324.830 13,6% 7,5%
(5% ≤ PD ≤ 20%)
Estágio 1 5.837.116 96,3% 490.321 96,1% 8,4% 4.170.990 96,9% 315.603 97,2% 7,6%
Estágio 2 226.770 3,7% 19.438 3,9% 8,6% 133.072 3,1% 9.227 2,8% 6,9%
Risco maior
4.761.732 23,8% 2.601.176 78,7% 54,6% 3.670.330 25,1% 1.938.295 81,1% 52,8%
(PD > 20%)
Estágio 1 990.988 20,8% 191.305 7,4% 19,4% 1.049.233 28,6% 229.234 11,8% 21,8%
Estágio 2 1.863.300 39,1% 796.156 30,6% 42,7% 1.228.767 33,5% 436.515 22,5% 35,5%
Estágio 3 1.907.444 40,1% 1.61

### Retirando elementos desnecessários da extração

In [3]:
def limpar_linhas_estagio(linhas):

    resultado = []
    ignorar_ate_estagio = False

    PD_HEADERS = ("Forte", "Satisfatório", "Risco maior") # Define título

    for linha in linhas:

        if linha.startswith(PD_HEADERS): # ignora tudo, menos estágios
            ignorar_ate_estagio = True
            continue

        if ignorar_ate_estagio:
            if linha.startswith("Estágio"): # Define sessão que não deve ser excluída
                ignorar_ate_estagio = False
            else:
                continue

        if not linha.startswith("Estágio"):
            continue

        tokens = linha.split()

        # remove porcentagens
        tokens_sem_percent = [t for t in tokens if "%" not in t]

        # ignora "Estágio" e o número do estágio
        tokens_dados = tokens_sem_percent[2:]

        numeros = [
            t for t in tokens_dados
            if re.fullmatch(r"\d{1,3}(?:\.\d{3})*|–", t)
        ]
        # --------------------------------

        estagio = " ".join(tokens_sem_percent[:2])  # Estágio 1 / 2 / 3

        resultado.append(
            " ".join([estagio] + numeros)
        )

    return resultado



In [4]:
linhas_limpa = limpar_linhas_estagio(linhas)

for l in linhas_limpa:
    print(l)

Estágio 1 9.202.497 193.814 6.628.863 126.147
Estágio 2 12 – – – – 16.057 254
Estágio 1 5.837.116 490.321 4.170.990 315.603
Estágio 2 226.770 19.438 133.072 9.227
Estágio 1 990.988 191.305 1.049.233 229.234
Estágio 2 1.863.300 796.156 1.228.767 436.515
Estágio 3 1.907.444 1.613.715 1.392.330 1.272.546


### Extração do trimestre

In [5]:
from datetime import datetime

def datas_para_trimestres(texto):
    datas = re.findall(r"\d{2}/\d{2}/\d{4}", texto)

    trimestres = []

    for d in datas:
        dt = datetime.strptime(d, "%d/%m/%Y")
        trimestre = (dt.month - 1) // 3 + 1
        ano = str(dt.year)[-2:]
        trimestres.append(f"{trimestre}T{ano}")

    return trimestres

### Padrão fixo para colunas do dataframe

In [6]:
PD_PADRAO = [
    "Forte (PD < 5%)",
    "Forte (PD < 5%)",
    "Satisfatório (5% ≤ PD ≤ 20%)",
    "Satisfatório (5% ≤ PD ≤ 20%)",
    "Risco maior (PD > 20%)",
    "Risco maior (PD > 20%)",
    "Risco maior (PD > 20%)",
]

ESTAGIO_PADRAO = [1, 2, 1, 2, 1, 2, 3]

### Para as colunas de Exposição Bruta e PE

In [7]:
def parse_linha_estagio(linha):
    partes = linha.split()
    estagio = int(partes[1])

    numeros = partes[2:]

    def conv(x):
        if x == "–":
            return None
        return float(x.replace(".", "").replace(",", "."))

    numeros = [conv(x) for x in numeros]

    return {
        "estagio": estagio,
        "exp_bruta_atual": numeros[0] if len(numeros) > 0 else None,
        "pe_atual": numeros[1] if len(numeros) > 1 else None,
        "exp_bruta_anterior": numeros[2] if len(numeros) > 2 else None,
        "pe_anterior": numeros[3] if len(numeros) > 3 else None,
    }

In [8]:
def construir_dataframe(
    linhas_estagio,
    texto_pdf,
    banco="Nubank",
    produto="crédito"
):

    trimestre_atual, trimestre_anterior = datas_para_trimestres(texto_pdf)

    registros = []

    for i, linha in enumerate(linhas_estagio):

        dados = parse_linha_estagio(linha)

        # Linha do trimestre atual
        registros.append({
            "ano": trimestre_atual,
            "banco": banco.lower(),
            "produto": produto,
            "PD": PD_PADRAO[i],
            "Estágio": ESTAGIO_PADRAO[i],
            "Exposição Bruta": dados["exp_bruta_atual"],
            "PE": dados["pe_atual"]
        })

        # Linha do trimestre anterior
        registros.append({
            "ano": trimestre_anterior,
            "banco": banco.lower(),
            "produto": produto,
            "PD": PD_PADRAO[i],
            "Estágio": ESTAGIO_PADRAO[i],
            "Exposição Bruta": dados["exp_bruta_anterior"],
            "PE": dados["pe_anterior"]
        })

    df = pd.DataFrame(registros)

    return df


In [9]:
df = construir_dataframe(
    linhas_estagio=linhas_limpa,
    texto_pdf=trecho,
    banco="Nubank",
    produto="crédito"
)